In [6]:
## NEXT STEP, leaving out h_4.

# script.sage

from isogeny_chain_dim2 import *

from itertools import product

k = 4
p = 8*3^k - 1
F1 = GF(p)
R.<x> = F1[]
Fp.<om> = GF(p^2, modulus=x^2+x+1)
omega = om


# We first sample a random abelian surface as the image of a (3,3)-isogeny chain
# coming from E1 x E2, where E2 is isogenous to E1

while True:
    E1 = EllipticCurve(Fp, [1,0])
    # "random" isogenous curve with the same product structure
    #P = E1.lift_x(2)
    P = 3^5*E1.random_element()
    E2 = E1.isogeny(P).codomain()
    
    # symplectic 3^k-torsion basis
    P1,P2,Q1,Q2 = create_basis(E1,E2,k,omega=omega)
    
    # random kernel for a (3^k,3^k)-isogeny
    a = ZZ.random_element(3^(k-1))
    b = ZZ.random_element(3^(k-2))
    c = ZZ.random_element(3^(k-1))
    b = 1 + 3*b # need b!=0, so that the first isogeny is non-diagonal.
    
    # (3^k,3^k)- group on (E1 x E2) in Hessian form (+ auxiliary information)
    # where the group is <(P1 + a*Q1, b*Q2),(b*Q1, P2 + c*Q2)
    (R,S),(R_9,S_9) = translate_to_Hessian((P1,P2,Q1,Q2),k,(a,b,c),E1,E2)
    
    A = R._parent
    
    Phi = compute_isogeny_chain((R,S), (R_9,S_9), k-1, (a,b,c))
    
    # we can push points lothrough the isogeny
    H2,H1 = A._elliptic_curves
    Rand1 = E1.random_element()
    Rand2 = E2.random_element()
    R1 = H1.map_point(Rand1)
    R2 = H2.map_point(Rand2)
    R12 = A([R2,R1]);
    phi_R12 = Phi(R12)
    
    # implicit test (note that addition on the Hessian is not implemented)
    # R12 + first kernel generator
    Test1 = Rand1 + 3*(P1 + a*Q1)
    Test2 = Rand2 + 3*b*Q2
    T1 = H1.map_point(Test1)
    T2 = H2.map_point(Test2)
    T12 = A([T2,T1])
    phi_T12 = Phi(T12)
    
    # R12 + second kernel generator
    Test1 = Rand1 + 3*(b*Q1)
    Test2 = Rand2 + 3*(P2 + c*Q2)
    S1 = H1.map_point(Test1)
    S2 = H2.map_point(Test2)
    S12 = A([S2,S1])
    phi_S12 = Phi(S12)
    
    if (phi_R12 == phi_S12 and phi_R12 == phi_T12):
        break

# We now interpolate the addition formulae on our randomly generated surface

def random_points():
    Rand1 = E1.random_element()
    Rand2 = E2.random_element()
    R1 = H1(Rand1)
    R2 = H2(Rand2)
    R12 = A([R2,R1])
    phi_R12 = Phi(R12)
    return (Rand1, Rand2, phi_R12)

def get_sample():
    while True:
        R1, R2, R12 = random_points()
        T1, T2, T12 = random_points()
        RT1 = R1 + T1
        RT2 = R2 + T2
        RT1 = H1(RT1)
        RT2 = H2(RT2)
        RT12 = A([RT2,RT1])
        phi_RT12 = Phi(RT12)
    
        if prod(R12)*prod(T12)*prod(phi_RT12) != 0:
            break
    
    return R12, T12, phi_RT12

def index_to_mons(index):
    i0 = index%3
    i1 = index//3
    mons = [(c00+3*c01,c10+3*c11)
            for c00 in range(3)
            for c01 in range(3)
            for c10 in range(3)
            for c11 in range(3)
            if (c00 + c10)%3 == i0
            and (c01 + c11)%3 == i1
            and 3*c11+c10 > 3*c01+c00]
    return mons
    
def square_mons(mons):
    mons = [mon1+mon2 for mon1 in mons for mon2 in mons]
    return mons

def get_monomials():
    monss = []
    for x in range(9):
        mons = index_to_mons(x)
        mons = square_mons(mons)
        monss.append(mons)
    return monss

monss = get_monomials()
ls = [len(mons) for mons in monss]

def zero_list(length):
    return [0 for _ in range(length)]

def get_rows(sample):
    R = sample[0]
    T = sample[1]
    RT = sample[2]
    rows = []
    for x in range(1,9):
        betweenzeros = sum([ls[it] for it in range(x-1)])
        postzeros = sum([ls[it] for it in range(x+1,9)])
            
        row = ([RT[x]*R[i]*R[j]*T[m]*T[n] for (i,j,m,n) in monss[0]]
               + zero_list(betweenzeros)
               + [-RT[0]*R[i]*R[j]*T[m]*T[n] for (i,j,m,n) in monss[x]]
               + zero_list(postzeros)
              )

        rows.append(row)

    return rows
    
M = []
for s in range(20):
    sample = get_sample()
    rows = get_rows(sample)
    M += rows
    
M = matrix(Fp, M)

K = M.right_kernel()

b0 = K.basis()[0]

assert K.dimension() == 1

B = Phi.codomain()

ts = B.zero()
hs = B._h
ds = B._d

In [16]:
monss[0]

[(3, 6, 3, 6),
 (3, 6, 1, 2),
 (3, 6, 4, 8),
 (3, 6, 5, 7),
 (1, 2, 3, 6),
 (1, 2, 1, 2),
 (1, 2, 4, 8),
 (1, 2, 5, 7),
 (4, 8, 3, 6),
 (4, 8, 1, 2),
 (4, 8, 4, 8),
 (4, 8, 5, 7),
 (5, 7, 3, 6),
 (5, 7, 1, 2),
 (5, 7, 4, 8),
 (5, 7, 5, 7)]

In [17]:
monss[1]

[(0, 1, 0, 1),
 (0, 1, 3, 7),
 (0, 1, 4, 6),
 (0, 1, 5, 8),
 (3, 7, 0, 1),
 (3, 7, 3, 7),
 (3, 7, 4, 6),
 (3, 7, 5, 8),
 (4, 6, 0, 1),
 (4, 6, 3, 7),
 (4, 6, 4, 6),
 (4, 6, 5, 8),
 (5, 8, 0, 1),
 (5, 8, 3, 7),
 (5, 8, 4, 6),
 (5, 8, 5, 8)]

In [21]:
for i in range(16):
    print()

1 (3, 6, 3, 6)
97*om + 213 (3, 6, 1, 2)
193*om + 521 (3, 6, 4, 8)
239*om + 259 (3, 6, 5, 7)
97*om + 213 (1, 2, 3, 6)
361*om + 551 (1, 2, 1, 2)
223*om + 211 (1, 2, 4, 8)
370*om + 316 (1, 2, 5, 7)
193*om + 521 (4, 8, 3, 6)
223*om + 211 (4, 8, 1, 2)
488*om + 148 (4, 8, 4, 8)
467*om + 541 (4, 8, 5, 7)
239*om + 259 (5, 7, 3, 6)
370*om + 316 (5, 7, 1, 2)
467*om + 541 (5, 7, 4, 8)
189*om + 414 (5, 7, 5, 7)


In [23]:
for i in range(16):
    print(b0[i], monss[0][i], b0[i+16], monss[1][i])

1 (3, 6, 3, 6) 361*om + 551 (0, 1, 0, 1)
97*om + 213 (3, 6, 1, 2) 223*om + 211 (0, 1, 3, 7)
193*om + 521 (3, 6, 4, 8) 370*om + 316 (0, 1, 4, 6)
239*om + 259 (3, 6, 5, 7) 97*om + 213 (0, 1, 5, 8)
97*om + 213 (1, 2, 3, 6) 223*om + 211 (3, 7, 0, 1)
361*om + 551 (1, 2, 1, 2) 488*om + 148 (3, 7, 3, 7)
223*om + 211 (1, 2, 4, 8) 467*om + 541 (3, 7, 4, 6)
370*om + 316 (1, 2, 5, 7) 193*om + 521 (3, 7, 5, 8)
193*om + 521 (4, 8, 3, 6) 370*om + 316 (4, 6, 0, 1)
223*om + 211 (4, 8, 1, 2) 467*om + 541 (4, 6, 3, 7)
488*om + 148 (4, 8, 4, 8) 189*om + 414 (4, 6, 4, 6)
467*om + 541 (4, 8, 5, 7) 239*om + 259 (4, 6, 5, 8)
239*om + 259 (5, 7, 3, 6) 97*om + 213 (5, 8, 0, 1)
370*om + 316 (5, 7, 1, 2) 193*om + 521 (5, 8, 3, 7)
467*om + 541 (5, 7, 4, 8) 239*om + 259 (5, 8, 4, 6)
189*om + 414 (5, 7, 5, 7) 1 (5, 8, 5, 8)


In [25]:
for i in range(16):
    print(monss[3][i], monss[4][i], monss[5][i])

(0, 3, 0, 3) (0, 4, 0, 4) (0, 5, 0, 5)
(0, 3, 1, 5) (0, 4, 6, 7) (0, 5, 6, 8)
(0, 3, 7, 8) (0, 4, 1, 3) (0, 5, 1, 4)
(0, 3, 2, 4) (0, 4, 2, 5) (0, 5, 2, 3)
(1, 5, 0, 3) (6, 7, 0, 4) (6, 8, 0, 5)
(1, 5, 1, 5) (6, 7, 6, 7) (6, 8, 6, 8)
(1, 5, 7, 8) (6, 7, 1, 3) (6, 8, 1, 4)
(1, 5, 2, 4) (6, 7, 2, 5) (6, 8, 2, 3)
(7, 8, 0, 3) (1, 3, 0, 4) (1, 4, 0, 5)
(7, 8, 1, 5) (1, 3, 6, 7) (1, 4, 6, 8)
(7, 8, 7, 8) (1, 3, 1, 3) (1, 4, 1, 4)
(7, 8, 2, 4) (1, 3, 2, 5) (1, 4, 2, 3)
(2, 4, 0, 3) (2, 5, 0, 4) (2, 3, 0, 5)
(2, 4, 1, 5) (2, 5, 6, 7) (2, 3, 6, 8)
(2, 4, 7, 8) (2, 5, 1, 3) (2, 3, 1, 4)
(2, 4, 2, 4) (2, 5, 2, 5) (2, 3, 2, 3)


In [2]:
_,test = get_meta_sample()

In [5]:
[[i, test[i]] for i in range(17)]

[[0, 1],
 [1, 48*om + 471],
 [2, 428*om + 605],
 [3, 413*om + 451],
 [4, 48*om + 471],
 [5, 61*om + 57],
 [6, 519*om + 230],
 [7, 215*om + 489],
 [8, 428*om + 605],
 [9, 519*om + 230],
 [10, 267*om + 486],
 [11, 182*om + 112],
 [12, 413*om + 451],
 [13, 215*om + 489],
 [14, 182*om + 112],
 [15, 549*om + 91],
 [16, 61*om + 57]]

In [2]:
# New interpolation, using the t_i's

from itertools import combinations_with_replacement

idxs = list(combinations_with_replacement(range(5), 6))

ls = len(idxs)
print(ls)

def zero_list(length):
    return [0 for _ in range(length)]

# NOW ONLY THE FIRST COEFFICIENT

def get_row(sample, index):
    ts, cs = sample
    assert cs[0] == 1
    ts = [ts[0], ts[1], ts[3], ts[4], ts[5]]
    
    monomials = [prod(ts[i] for i in idx) for idx in idxs]
    
    row = (
        [cs[index] * m for m in monomials] +
        [-cs[0] * m for m in monomials]
    )
    
    return row

210


In [ ]:
index_list = [1,2,3,5,6,7,10,11,16]

matrix_list = [[] for _ in range(17)]
for s in range(450):
    sample = get_meta_sample()
    for index in index_list:
        row = get_row(sample, index)
        matrix_list[index].append(row)

basis_list = []

for mat in matrix_list:
    if mat:
        M = matrix(Fp, mat)
        # compute kernel
        K = M.right_kernel()
        print(K)
        bas = K.basis()[0]
        basis_list.append(bas)

In [ ]:
R.<t0,t1,t3,t4,t5> = Fp[]

vars = [t0,t1,t3,t4,t5]

mon_list = [prod(vars[i] for i in idx) for idx in idxs]

for bas in basis_list:
    first = [bas[i] for i in range(210)]
    second = [bas[i] for i in range(210,420)]
    print(vec1.inner_product(vec_mon), vec2.inner_product(vec_mon))

In [ ]:
# We now interpolate global addition formulae, by interpolating over Abelian surfaces

# Precompute monomial indices

# We leave out all monomials containing h_4, since d_4 = 1 and sum h_id_i = 0.
idxs = [
    (i1, j1, k1, i2, j2, k2)
    for i1 in range(4) for j1 in range(i1, 4) for k1 in range(j1, 4)
    for i2 in range(5) for j2 in range(i2, 5) for k2 in range(j2, 5)
]
ls = len(idxs)
print(ls)

def zero_list(length):
    return [0 for _ in range(length)]

# NOW ONLY THE FIRST COEFFICIENT

def get_row(sample):
    hs, ds, cs = sample
    assert cs[0] == 1
    assert hs[0] == 1
    ds = [d for d in ds] + [1]
    
    row = ([cs[1]*hs[i1]*hs[j1]*hs[k1]*ds[i2]*ds[j2]*ds[k2] for (i1, j1, k1, i2, j2, k2) in idxs]
           + [-cs[0]*hs[i1]*hs[j1]*hs[k1]*ds[i2]*ds[j2]*ds[k2] for (i1, j1, k1, i2, j2, k2) in idxs]
          )
    return row

# construct interpolation matrix
# Now we filtered out some trivial relations
M = []
for s in range(1):
    try:
        sample = get_meta_sample()
        row = get_row(sample)
        M.append(row)
    except:
        print('failure')
    
M = matrix(Fp, M)

# compute kernel
K = M.right_kernel()

print(K)

bas = K.basis()[0]

In [ ]:
import ast

with open("basis_new.txt") as f:
    data = ast.literal_eval(f.read())

idxs = [
    (i1, j1, k1, i2, j2, k2)
    for i1 in range(4) for j1 in range(i1, 4) for k1 in range(j1, 4)
    for i2 in range(5) for j2 in range(i2, 5) for k2 in range(j2, 5)
]

basis_vector = data[4]

relation = [(basis_vector[i], idxs[i]) for i in range(len(idxs)) if basis_vector[i] != 0]

relation

In [ ]:
hs, ds, cs = get_meta_sample()
assert cs[0] == 1
assert hs[0] == 1
ds = [d for d in ds] + [1]

result = 0

for mon in relation:
    c = mon[0]
    (i1, j1, k1, i2, j2, k2) = mon[1]
    result += c*hs[i1]*hs[j1]*hs[k1]*ds[i2]*ds[j2]*ds[k2]

result

In [ ]:
len(cs)

In [ ]:
for d in data:
    print([index for index in range(2*len(idxs)) if d[index] != 0])

In [ ]:
hs,ds

In [ ]:
hs, ds, cs = get_meta_sample()

In [ ]:
hs,ds

In [ ]:
def add(P, Q):
    A = P._parent
    assert Q._parent == P._parent

    A0, A1, A2, A3, A4, A5, A6, A7, A8 = P.coordinates()
    B0, B1, B2, B3, B4, B5, B6, B7, B8 = Q.coordinates()

    X0 = A4*A8*B0^2 - A3*A6*B1*B2 - A1*A2*B3*B6 + A0^2*B4*B8
    X1 = -A5*A8*B0*B1 + A3*A7*B2^2 + A2^2*B3*B7 - A0*A1*B5*B8
    X2 = A3*A8*B1^2 - A4*A7*B0*B2 - A0*A2*B4*B7 + A1^2*B3*B8
    X3 = -A7*A8*B0*B3 + A6^2*B1*B5 + A1*A5*B6^2 - A0*A3*B7*B8
    X4 = A8^2*B0*B4 - A6*A7*B2*B5 - A2*A5*B6*B7 + A0*A4*B8^2
    X5 = -A6*A8*B1*B4 + A7^2*B0*B5 + A0*A5*B7^2 - A1*A4*B6*B8
    X6 = A1*A8*B3^2 - A0*A6*B4*B5 - A4*A5*B0*B6 + A3^2*B1*B8
    X7 = -A2*A8*B3*B4 + A0*A7*B5^2 + A5^2*B0*B7 - A3*A4*B2*B8
    X8 = A0*A8*B4^2 - A1*A7*B3*B5 - A3*A5*B1*B7 + A4^2*B0*B8

    return A([X0, X1, X2, X3, X4, X5, X6, X7, X8])

In [ ]:
len(b0)

In [ ]:
b0

In [ ]:
from isogeny_chain_dim2 import *

k = 4
p = 8*3^k - 1
F1 = GF(p)
R.<x> = F1[]
Fp.<om> = GF(p^2, modulus=x^2+x+1)
omega = om

while True:
    E1 = EllipticCurve(Fp, [1,0])
    # "random" isogenous curve with the same product structure
    #P = E1.lift_x(2)
    P = 3^5*E1.random_element()
    E2 = E1.isogeny(P).codomain()
    
    # symplectic 3^k-torsion basis
    P1,P2,Q1,Q2 = create_basis(E1,E2,k,omega=omega)
    
    # random kernel for a (3^k,3^k)-isogeny
    a = ZZ.random_element(3^(k-1))
    b = ZZ.random_element(3^(k-2))
    c = ZZ.random_element(3^(k-1))
    b = 1 + 3*b # need b!=0, so that the first isogeny is non-diagonal.
    
    # (3^k,3^k)- group on (E1 x E2) in Hessian form (+ auxiliary information)
    # where the group is <(P1 + a*Q1, b*Q2),(b*Q1, P2 + c*Q2)
    (R,S),(R_9,S_9) = translate_to_Hessian((P1,P2,Q1,Q2),k,(a,b,c),E1,E2)
    
    A = R._parent
    
    Phi = compute_isogeny_chain((R,S), (R_9,S_9), k-1, (a,b,c))
    
    # we can push points lothrough the isogeny
    H2,H1 = A._elliptic_curves
    Rand1 = E1.random_element()
    Rand2 = E2.random_element()
    R1 = H1.map_point(Rand1)
    R2 = H2.map_point(Rand2)
    R12 = A([R2,R1]);
    phi_R12 = Phi(R12)
    
    # implicit test (note that addition on the Hessian is not implemented)
    # R12 + first kernel generator
    Test1 = Rand1 + 3*(P1 + a*Q1)
    Test2 = Rand2 + 3*b*Q2
    T1 = H1.map_point(Test1)
    T2 = H2.map_point(Test2)
    T12 = A([T2,T1])
    phi_T12 = Phi(T12)
    
    # R12 + second kernel generator
    Test1 = Rand1 + 3*(b*Q1)
    Test2 = Rand2 + 3*(P2 + c*Q2)
    S1 = H1.map_point(Test1)
    S2 = H2.map_point(Test2)
    S12 = A([S2,S1])
    phi_S12 = Phi(S12)
    
    if (phi_R12 == phi_S12 and phi_R12 == phi_T12):
        break

B = Phi.codomain()

hs = B._h
ds = B._d

ds =  [d for d in ds] + [1]

sum([hs[x]*ds[x] for x in range(5)]) == 0

In [ ]:
idxs = [
    (i1, j1, k1, i2, j2, k2)
    for i1 in range(5) for j1 in range(i1, 5) for k1 in range(j1, 5)
    for i2 in range(5) for j2 in range(i2, 5) for k2 in range(j2, 5)
]
ls = len(idxs)
print(ls)

In [ ]:
from isogeny_chain_dim2 import *

def get_meta_sample():
    ####################################
    ####################################
    ## trying addition formulas
    
    from itertools import product
    
    k = 4
    p = 8*3^k - 1
    F1 = GF(p)
    R.<x> = F1[]
    Fp.<om> = GF(p^2, modulus=x^2+x+1)
    omega = om
    
    while True:
        E1 = EllipticCurve(Fp, [1,0])
        # "random" isogenous curve with the same product structure
        #P = E1.lift_x(2)
        P = 3^5*E1.random_element()
        E2 = E1.isogeny(P).codomain()
        
        # symplectic 3^k-torsion basis
        P1,P2,Q1,Q2 = create_basis(E1,E2,k,omega=omega)
        
        # random kernel for a (3^k,3^k)-isogeny
        a = ZZ.random_element(3^(k-1))
        b = ZZ.random_element(3^(k-2))
        c = ZZ.random_element(3^(k-1))
        b = 1 + 3*b # need b!=0, so that the first isogeny is non-diagonal.
        
        # (3^k,3^k)- group on (E1 x E2) in Hessian form (+ auxiliary information)
        # where the group is <(P1 + a*Q1, b*Q2),(b*Q1, P2 + c*Q2)
        (R,S),(R_9,S_9) = translate_to_Hessian((P1,P2,Q1,Q2),k,(a,b,c),E1,E2)
        
        A = R._parent
        
        Phi = compute_isogeny_chain((R,S), (R_9,S_9), k-1, (a,b,c))
        
        # we can push points lothrough the isogeny
        H2,H1 = A._elliptic_curves
        Rand1 = E1.random_element()
        Rand2 = E2.random_element()
        R1 = H1.map_point(Rand1)
        R2 = H2.map_point(Rand2)
        R12 = A([R2,R1]);
        phi_R12 = Phi(R12)
        
        # implicit test (note that addition on the Hessian is not implemented)
        # R12 + first kernel generator
        Test1 = Rand1 + 3*(P1 + a*Q1)
        Test2 = Rand2 + 3*b*Q2
        T1 = H1.map_point(Test1)
        T2 = H2.map_point(Test2)
        T12 = A([T2,T1])
        phi_T12 = Phi(T12)
        
        # R12 + second kernel generator
        Test1 = Rand1 + 3*(b*Q1)
        Test2 = Rand2 + 3*(P2 + c*Q2)
        S1 = H1.map_point(Test1)
        S2 = H2.map_point(Test2)
        S12 = A([S2,S1])
        phi_S12 = Phi(S12)
        
        if (phi_R12 == phi_S12 and phi_R12 == phi_T12):
            break
    
    def random_points():
        Rand1 = E1.random_element()
        Rand2 = E2.random_element()
        R1 = H1(Rand1)
        R2 = H2(Rand2)
        R12 = A([R2,R1])
        phi_R12 = Phi(R12)
        return (Rand1, Rand2, phi_R12)
    
    # def get_sample():
    #     Rand1 = E1.random_element()
    #     Rand2 = E2.random_element()
    #     R1 = H1(Rand1)
    #     R2 = H2(Rand2)
    #     R12 = A([R2,R1])
        
    #     Tand1 = E1.random_element()
    #     Tand2 = E2.random_element()
    #     T1 = H1(Tand1)
    #     T2 = H2(Tand2)
    #     T12 = A([T2,T1])
        
    #     RT1 = H1(Rand1 + Tand1)
    #     RT2 = H2(Rand2 + Tand2)
        
    #     RT12 = A([RT2,RT1])
        
    #     R12 = A([R2,R1])
    #     T12 = A([T2,T1])
    
    #     return R12, T12, RT12
    
    def get_sample():
        while True:
            R1, R2, R12 = random_points()
            T1, T2, T12 = random_points()
            RT1 = R1 + T1
            RT2 = R2 + T2
            RT1 = H1(RT1)
            RT2 = H2(RT2)
            RT12 = A([RT2,RT1])
            phi_RT12 = Phi(RT12)
        
            if prod(R12)*prod(T12)*prod(phi_RT12) != 0:
                break
        
        return R12, T12, phi_RT12
    
    def index_to_mons(index):
        i0 = index%3
        i1 = index//3
        mons = [(c00+3*c01,c10+3*c11)
                for c00 in range(3)
                for c01 in range(3)
                for c10 in range(3)
                for c11 in range(3)
                if (c00 + c10)%3 == i0
                and (c01 + c11)%3 == i1
                and 3*c11+c10 > 3*c01+c00]
        return mons
        
    def square_mons(mons):
        mons = [mon1+mon2 for mon1 in mons for mon2 in mons]
        return mons
    
    def get_monomials():
        monss = []
        for x in range(9):
            mons = index_to_mons(x)
            mons = square_mons(mons)
            monss.append(mons)
        return monss
    
    monss = get_monomials()
    ls = [len(mons) for mons in monss]
    
    def zero_list(length):
        return [0 for _ in range(length)]
    
    def get_rows(sample):
        R = sample[0]
        T = sample[1]
        RT = sample[2]
        rows = []
        for x in range(1,9):
            betweenzeros = sum([ls[it] for it in range(x-1)])
            postzeros = sum([ls[it] for it in range(x+1,9)])
                
            row = ([RT[x]*R[i]*R[j]*T[m]*T[n] for (i,j,m,n) in monss[0]]
                   + zero_list(betweenzeros)
                   + [-RT[0]*R[i]*R[j]*T[m]*T[n] for (i,j,m,n) in monss[x]]
                   + zero_list(postzeros)
                  )
    
            rows.append(row)
    
        return rows
        
    M = []
    for s in range(20):
        sample = get_sample()
        rows = get_rows(sample)
        M += rows
        
    M = matrix(Fp, M)
    
    b0 = M.right_kernel().basis()[0]
    
    B = Phi.codomain()
    
    hs = B._h
    ds = B._d
    
    return hs,ds,b0[0:17]

ls = len([1 for i1 in range(5) for j1 in range(i1,5) for i2 in range(5) for j2 in range(i2,5)])

print(ls)

def zero_list(length):
    return [0 for _ in range(length)]

def get_rows(sample):
    hs = sample[0]
    ds = sample[1]
    cs = sample[2]
    rows = []
    
    for x in range(1,17):
        betweenzeros = sum([ls for it in range(x-1)])
        postzeros = sum([ls for it in range(x+1,17)])
            
        row = ([cs[x]*hs[i1]*hs[j1]*ds[i2]*ds[j2] for i1 in range(5) for j1 in range(i1,5) for i2 in range(4) for j2 in range(i2,4)]
               + zero_list(betweenzeros)
               + [-cs[0]*hs[i1]*hs[j1]*ds[i2]*ds[j2] for i1 in range(5) for j1 in range(i1,5) for i2 in range(4) for j2 in range(i2,4)]
               + zero_list(postzeros)
              )

        rows.append(row)

    return rows

M = []
for s in range(1):
    try:
        sample = get_meta_sample()
        rows = get_rows(sample)
        M += rows
    except:
        True
    
M = matrix(Fp, M)

M

In [ ]:
# script.sage

from isogeny_chain_dim2 import *

from itertools import product

k = 4
p = 8*3^k - 1
F1 = GF(p)
R.<x> = F1[]
Fp.<om> = GF(p^2, modulus=x^2+x+1)
omega = om


def get_meta_sample():

    # We first sample a random abelian surface as the image of a (3,3)-isogeny chain
    # coming from E1 x E2, where E2 is isogenous to E1
    
    while True:
        E1 = EllipticCurve(Fp, [1,0])
        # "random" isogenous curve with the same product structure
        #P = E1.lift_x(2)
        P = 3^5*E1.random_element()
        E2 = E1.isogeny(P).codomain()
        
        # symplectic 3^k-torsion basis
        P1,P2,Q1,Q2 = create_basis(E1,E2,k,omega=omega)
        
        # random kernel for a (3^k,3^k)-isogeny
        a = ZZ.random_element(3^(k-1))
        b = ZZ.random_element(3^(k-2))
        c = ZZ.random_element(3^(k-1))
        b = 1 + 3*b # need b!=0, so that the first isogeny is non-diagonal.
        
        # (3^k,3^k)- group on (E1 x E2) in Hessian form (+ auxiliary information)
        # where the group is <(P1 + a*Q1, b*Q2),(b*Q1, P2 + c*Q2)
        (R,S),(R_9,S_9) = translate_to_Hessian((P1,P2,Q1,Q2),k,(a,b,c),E1,E2)
        
        A = R._parent
        
        Phi = compute_isogeny_chain((R,S), (R_9,S_9), k-1, (a,b,c))
        
        # we can push points lothrough the isogeny
        H2,H1 = A._elliptic_curves
        Rand1 = E1.random_element()
        Rand2 = E2.random_element()
        R1 = H1.map_point(Rand1)
        R2 = H2.map_point(Rand2)
        R12 = A([R2,R1]);
        phi_R12 = Phi(R12)
        
        # implicit test (note that addition on the Hessian is not implemented)
        # R12 + first kernel generator
        Test1 = Rand1 + 3*(P1 + a*Q1)
        Test2 = Rand2 + 3*b*Q2
        T1 = H1.map_point(Test1)
        T2 = H2.map_point(Test2)
        T12 = A([T2,T1])
        phi_T12 = Phi(T12)
        
        # R12 + second kernel generator
        Test1 = Rand1 + 3*(b*Q1)
        Test2 = Rand2 + 3*(P2 + c*Q2)
        S1 = H1.map_point(Test1)
        S2 = H2.map_point(Test2)
        S12 = A([S2,S1])
        phi_S12 = Phi(S12)
        
        if (phi_R12 == phi_S12 and phi_R12 == phi_T12):
            break

    # We now interpolate the addition formulae on our randomly generated surface
    
    def random_points():
        Rand1 = E1.random_element()
        Rand2 = E2.random_element()
        R1 = H1(Rand1)
        R2 = H2(Rand2)
        R12 = A([R2,R1])
        phi_R12 = Phi(R12)
        return (Rand1, Rand2, phi_R12)
    
    def get_sample():
        while True:
            R1, R2, R12 = random_points()
            T1, T2, T12 = random_points()
            RT1 = R1 + T1
            RT2 = R2 + T2
            RT1 = H1(RT1)
            RT2 = H2(RT2)
            RT12 = A([RT2,RT1])
            phi_RT12 = Phi(RT12)
        
            if prod(R12)*prod(T12)*prod(phi_RT12) != 0:
                break
        
        return R12, T12, phi_RT12
    
    def index_to_mons(index):
        i0 = index%3
        i1 = index//3
        mons = [(c00+3*c01,c10+3*c11)
                for c00 in range(3)
                for c01 in range(3)
                for c10 in range(3)
                for c11 in range(3)
                if (c00 + c10)%3 == i0
                and (c01 + c11)%3 == i1
                and 3*c11+c10 > 3*c01+c00]
        return mons
        
    def square_mons(mons):
        mons = [mon1+mon2 for mon1 in mons for mon2 in mons]
        return mons
    
    def get_monomials():
        monss = []
        for x in range(9):
            mons = index_to_mons(x)
            mons = square_mons(mons)
            monss.append(mons)
        return monss
    
    monss = get_monomials()
    ls = [len(mons) for mons in monss]
    
    def zero_list(length):
        return [0 for _ in range(length)]
    
    def get_rows(sample):
        R = sample[0]
        T = sample[1]
        RT = sample[2]
        rows = []
        for x in range(1,9):
            betweenzeros = sum([ls[it] for it in range(x-1)])
            postzeros = sum([ls[it] for it in range(x+1,9)])
                
            row = ([RT[x]*R[i]*R[j]*T[m]*T[n] for (i,j,m,n) in monss[0]]
                   + zero_list(betweenzeros)
                   + [-RT[0]*R[i]*R[j]*T[m]*T[n] for (i,j,m,n) in monss[x]]
                   + zero_list(postzeros)
                  )
    
            rows.append(row)
    
        return rows
        
    M = []
    for s in range(20):
        sample = get_sample()
        rows = get_rows(sample)
        M += rows
        
    M = matrix(Fp, M)

    K = M.right_kernel()
    
    b0 = K.basis()[0]
    
    B = Phi.codomain()
    
    hs = B._h
    ds = B._dfrom isogeny_chain_dim2 import *

from itertools import product

k = 4
p = 8*3^k - 1
F1 = GF(p)
R.<x> = F1[]
Fp.<om> = GF(p^2, modulus=x^2+x+1)
omega = om


def get_meta_sample():

    # We first sample a random abelian surface as the image of a (3,3)-isogeny chain
    # coming from E1 x E2, where E2 is isogenous to E1
    
    while True:
        E1 = EllipticCurve(Fp, [1,0])
        # "random" isogenous curve with the same product structure
        #P = E1.lift_x(2)
        P = 3^5*E1.random_element()
        E2 = E1.isogeny(P).codomain()
        
        # symplectic 3^k-torsion basis
        P1,P2,Q1,Q2 = create_basis(E1,E2,k,omega=omega)
        
        # random kernel for a (3^k,3^k)-isogeny
        a = ZZ.random_element(3^(k-1))
        b = ZZ.random_element(3^(k-2))
        c = ZZ.random_element(3^(k-1))
        b = 1 + 3*b # need b!=0, so that the first isogeny is non-diagonal.
        
        # (3^k,3^k)- group on (E1 x E2) in Hessian form (+ auxiliary information)
        # where the group is <(P1 + a*Q1, b*Q2),(b*Q1, P2 + c*Q2)
        (R,S),(R_9,S_9) = translate_to_Hessian((P1,P2,Q1,Q2),k,(a,b,c),E1,E2)
        
        A = R._parent
        
        Phi = compute_isogeny_chain((R,S), (R_9,S_9), k-1, (a,b,c))
        
        # we can push points lothrough the isogeny
        H2,H1 = A._elliptic_curves
        Rand1 = E1.random_element()
        Rand2 = E2.random_element()
        R1 = H1.map_point(Rand1)
        R2 = H2.map_point(Rand2)
        R12 = A([R2,R1]);
        phi_R12 = Phi(R12)
        
        # implicit test (note that addition on the Hessian is not implemented)
        # R12 + first kernel generator
        Test1 = Rand1 + 3*(P1 + a*Q1)
        Test2 = Rand2 + 3*b*Q2
        T1 = H1.map_point(Test1)
        T2 = H2.map_point(Test2)
        T12 = A([T2,T1])
        phi_T12 = Phi(T12)
        
        # R12 + second kernel generator
        Test1 = Rand1 + 3*(b*Q1)
        Test2 = Rand2 + 3*(P2 + c*Q2)
        S1 = H1.map_point(Test1)
        S2 = H2.map_point(Test2)
        S12 = A([S2,S1])
        phi_S12 = Phi(S12)
        
        if (phi_R12 == phi_S12 and phi_R12 == phi_T12):
            break

    # We now interpolate the addition formulae on our randomly generated surface
    
    def random_points():
        Rand1 = E1.random_element()
        Rand2 = E2.random_element()
        R1 = H1(Rand1)
        R2 = H2(Rand2)
        R12 = A([R2,R1])
        phi_R12 = Phi(R12)
        return (Rand1, Rand2, phi_R12)
    
    def get_sample():
        while True:
            R1, R2, R12 = random_points()
            T1, T2, T12 = random_points()
            RT1 = R1 + T1
            RT2 = R2 + T2
            RT1 = H1(RT1)
            RT2 = H2(RT2)
            RT12 = A([RT2,RT1])
            phi_RT12 = Phi(RT12)
        
            if prod(R12)*prod(T12)*prod(phi_RT12) != 0:
                break
        
        return R12, T12, phi_RT12
    
    def index_to_mons(index):
        i0 = index%3
        i1 = index//3
        mons = [(c00+3*c01,c10+3*c11)
                for c00 in range(3)
                for c01 in range(3)
                for c10 in range(3)
                for c11 in range(3)
                if (c00 + c10)%3 == i0
                and (c01 + c11)%3 == i1
                and 3*c11+c10 > 3*c01+c00]
        return mons
        
    def square_mons(mons):
        mons = [mon1+mon2 for mon1 in mons for mon2 in mons]
        return mons
    
    def get_monomials():
        monss = []
        for x in range(9):
            mons = index_to_mons(x)
            mons = square_mons(mons)
            monss.append(mons)
        return monss
    
    monss = get_monomials()
    ls = [len(mons) for mons in monss]
    
    def zero_list(length):
        return [0 for _ in range(length)]
    
    def get_rows(sample):
        R = sample[0]
        T = sample[1]
        RT = sample[2]
        rows = []
        for x in range(1,9):
            betweenzeros = sum([ls[it] for it in range(x-1)])
            postzeros = sum([ls[it] for it in range(x+1,9)])
                
            row = ([RT[x]*R[i]*R[j]*T[m]*T[n] for (i,j,m,n) in monss[0]]
                   + zero_list(betweenzeros)
                   + [-RT[0]*R[i]*R[j]*T[m]*T[n] for (i,j,m,n) in monss[x]]
                   + zero_list(postzeros)
                  )
    
            rows.append(row)
    
        return rows
        
    M = []
    for s in range(20):
        sample = get_sample()
        rows = get_rows(sample)
        M += rows
        
    M = matrix(Fp, M)

    K = M.right_kernel()
    
    b0 = K.basis()[0]
    
    B = Phi.codomain()
    
    hs = B._h
    ds = B._d
    
    return hs,ds,b0[0:17]

    
    return hs,ds,b0[0:17]

# We now interpolate global addition formulae, by interpolating over Abelian surfaces

# Precompute monomial indices
idxs = [
    (i1, j1, i2, j2)
    for i1 in range(5) for j1 in range(i1, 5)
    for i2 in range(4) for j2 in range(i2, 4)
]
ls = len(idxs)

def zero_list(length):
    return [0 for _ in range(length)]

def get_rows(sample):
    hs, ds, cs = sample
    
    rows = []
    
    for x in range(1,17):
        betweenzeros = sum([ls for it in range(x-1)])
        postzeros = sum([ls for it in range(x+1,17)])
            
        row = ([cs[x]*hs[i1]*hs[j1]*ds[i2]*ds[j2] for (i1, j1, i2, j2) in idxs]
               + zero_list(betweenzeros)
               + [-cs[0]*hs[i1]*hs[j1]*ds[i2]*ds[j2] for (i1, j1, i2, j2) in idxs]
               + zero_list(postzeros)
              )

        rows.append(row)

    return rows

# construct interpolation matrix
M = []
for s in range(1):
    try:
        sample = get_meta_sample()
        rows = get_rows(sample)
        M += rows
    except:
        True
    
M = matrix(Fp, M)

# compute kernel
K = M.right_kernel()

print(K)

bas = K.basis()[0]

with open('basis.txt', 'w') as op:
    op.write(str(bas))

In [ ]:
# Replaced field Fp by F

from isogeny_chain_dim2 import *

from itertools import product

k = 4
p = 8*3^k - 1
F1 = GF(p)
R.<x> = F1[]
F.<om> = GF(p^2, modulus=x^2+x+1)
omega = om


def get_meta_sample():

    # We first sample a random abelian surface as the image of a (3,3)-isogeny chain
    # coming from E1 x E2, where E2 is isogenous to E1
    
    while True:
        E1 = EllipticCurve(Fp, [1,0])
        # "random" isogenous curve with the same product structure
        #P = E1.lift_x(2)
        P = 3^5*E1.random_element()
        E2 = E1.isogeny(P).codomain()
        
        # symplectic 3^k-torsion basis
        P1,P2,Q1,Q2 = create_basis(E1,E2,k,omega=omega)
        
        # random kernel for a (3^k,3^k)-isogeny
        a = ZZ.random_element(3^(k-1))
        b = ZZ.random_element(3^(k-2))
        c = ZZ.random_element(3^(k-1))
        b = 1 + 3*b # need b!=0, so that the first isogeny is non-diagonal.
        
        # (3^k,3^k)- group on (E1 x E2) in Hessian form (+ auxiliary information)
        # where the group is <(P1 + a*Q1, b*Q2),(b*Q1, P2 + c*Q2)
        (R,S),(R_9,S_9) = translate_to_Hessian((P1,P2,Q1,Q2),k,(a,b,c),E1,E2)
        
        A = R._parent
        
        Phi = compute_isogeny_chain((R,S), (R_9,S_9), k-1, (a,b,c))
        
        # we can push points lothrough the isogeny
        H2,H1 = A._elliptic_curves
        Rand1 = E1.random_element()
        Rand2 = E2.random_element()
        R1 = H1.map_point(Rand1)
        R2 = H2.map_point(Rand2)
        R12 = A([R2,R1]);
        phi_R12 = Phi(R12)
        
        # implicit test (note that addition on the Hessian is not implemented)
        # R12 + first kernel generator
        Test1 = Rand1 + 3*(P1 + a*Q1)
        Test2 = Rand2 + 3*b*Q2
        T1 = H1.map_point(Test1)
        T2 = H2.map_point(Test2)
        T12 = A([T2,T1])
        phi_T12 = Phi(T12)
        
        # R12 + second kernel generator
        Test1 = Rand1 + 3*(b*Q1)
        Test2 = Rand2 + 3*(P2 + c*Q2)
        S1 = H1.map_point(Test1)
        S2 = H2.map_point(Test2)
        S12 = A([S2,S1])
        phi_S12 = Phi(S12)
        
        if (phi_R12 == phi_S12 and phi_R12 == phi_T12):
            break

    # We now interpolate the addition formulae on our randomly generated surface
    
    def random_points():
        Rand1 = E1.random_element()
        Rand2 = E2.random_element()
        R1 = H1(Rand1)
        R2 = H2(Rand2)
        R12 = A([R2,R1])
        phi_R12 = Phi(R12)
        return (Rand1, Rand2, phi_R12)
    
    def get_sample():
        while True:
            R1, R2, R12 = random_points()
            T1, T2, T12 = random_points()
            RT1 = R1 + T1
            RT2 = R2 + T2
            RT1 = H1(RT1)
            RT2 = H2(RT2)
            RT12 = A([RT2,RT1])
            phi_RT12 = Phi(RT12)
        
            if prod(R12)*prod(T12)*prod(phi_RT12) != 0:
                break
        
        return R12, T12, phi_RT12
    
    def index_to_mons(index):
        i0 = index%3
        i1 = index//3
        mons = [(c00+3*c01,c10+3*c11)
                for c00 in range(3)
                for c01 in range(3)
                for c10 in range(3)
                for c11 in range(3)
                if (c00 + c10)%3 == i0
                and (c01 + c11)%3 == i1
                and 3*c11+c10 > 3*c01+c00]
        return mons
        
    def square_mons(mons):
        mons = [mon1+mon2 for mon1 in mons for mon2 in mons]
        return mons
    
    def get_monomials():
        monss = []
        for x in range(9):
            mons = index_to_mons(x)
            mons = square_mons(mons)
            monss.append(mons)
        return monss
    
    monss = get_monomials()
    ls = [len(mons) for mons in monss]
    
    def zero_list(length):
        return [0 for _ in range(length)]
    
    def get_rows(sample):
        R = sample[0]
        T = sample[1]
        RT = sample[2]
        rows = []
        for x in range(1,9):
            betweenzeros = sum([ls[it] for it in range(x-1)])
            postzeros = sum([ls[it] for it in range(x+1,9)])
                
            row = ([RT[x]*R[i]*R[j]*T[m]*T[n] for (i,j,m,n) in monss[0]]
                   + zero_list(betweenzeros)
                   + [-RT[0]*R[i]*R[j]*T[m]*T[n] for (i,j,m,n) in monss[x]]
                   + zero_list(postzeros)
                  )
    
            rows.append(row)
    
        return rows
        
    M = []
    for s in range(20):
        sample = get_sample()
        rows = get_rows(sample)
        M += rows
        
    M = matrix(Fp, M)

    K = M.right_kernel()
    
    b0 = K.basis()[0]
    
    B = Phi.codomain()
    
    hs = B._h
    ds = B._d
    
    return hs,ds,b0[0:17]


In [ ]:
# ChatGPT method

############################################
# 2. Monomial index set
#    h_i h_j d_k d_l with symmetry
############################################

idxs = [
    (i1, j1, i2, j2)
    for i1 in range(5) for j1 in range(i1, 5)
    for i2 in range(4) for j2 in range(i2, 4)
]
ls = len(idxs)   # number of monomials

############################################
# 3. Monomial vector for one sample
############################################

def monomial_vector(hs, ds):
    """
    Returns m in F^ls where
    m[k] = h[i1]*h[j1]*d[i2]*d[j2]
    """
    return vector(F, [
        hs[i1] * hs[j1] * ds[i2] * ds[j2]
        for (i1, j1, i2, j2) in idxs
    ])

############################################
# 4. Reduced interpolation matrix
#    (Kronecker factorization)
############################################

def reduced_matrix(samples):
    """
    Builds the small interpolation matrix of size
    (16 * #samples) x 17
    """
    rows = []

    for sample in samples:
        hs, ds, cs = sample

        # one row per x = 1..16
        for x in range(1, 17):
            row = [F(0)] * 17
            row[x] = cs[x]
            row[0] = -cs[0]
            rows.append(row)

    return Matrix(F, rows)

############################################
# 5. Kernel lifting
############################################

def lift_kernel_vector(w, m):
    """
    Given:
      w in F^17   (kernel vector of reduced system)
      m in F^ls   (monomial vector)

    Returns:
      v in F^(17*ls) lifting w
    """
    v = []
    for x in range(17):
        v.extend([w[x] * mk for mk in m])
    return vector(F, v)

############################################
# 6. Example usage
############################################

# Collect samples
num_samples = 200
samples = [get_meta_sample() for _ in range(num_samples)]

# Build reduced matrix
R = reduced_matrix(samples)

print("Reduced matrix size:", R.nrows(), "x", R.ncols())

# Compute kernel
K = R.right_kernel()
print("Reduced kernel dimension:", K.dimension())

# Lift kernel vectors
m0 = monomial_vector(samples[0][0], samples[0][1])
lifted_kernel = [lift_kernel_vector(w, m0) for w in K.basis()]

print("Lifted kernel vectors:", len(lifted_kernel))
print("Ambient dimension:", 17 * ls)

In [ ]:
# ChatGPT method 2

############################################
# 2. Monomial index set
############################################

idxs = [
    (i1, j1, i2, j2)
    for i1 in range(5) for j1 in range(i1, 5)
    for i2 in range(4) for j2 in range(i2, 4)
]
ls = len(idxs)          # monomials per block
NBLOCKS = 17
DIM = NBLOCKS * ls

############################################
# 3. Monomial vector
############################################

def monomial_vector(hs, ds):
    return vector(F, [
        hs[i1] * hs[j1] * ds[i2] * ds[j2]
        for (i1, j1, i2, j2) in idxs
    ])

############################################
# 4. Reduced kernel for ONE sample
############################################

def reduced_kernel(sample):
    """
    Returns a basis of the reduced kernel (subspace of F^17)
    """
    hs, ds, cs = sample
    rows = []

    for x in range(1, 17):
        row = [F(0)] * 17
        row[x] = cs[x]
        row[0] = -cs[0]
        rows.append(row)

    R = Matrix(F, rows)
    return R.right_kernel()

############################################
# 5. Lift reduced kernel to FULL constraints
############################################

def lifted_constraints(sample):
    """
    Returns a list of linear forms (rows) in F^(17*ls)
    encoding <v_x, m> = w_x for all w in reduced kernel
    """
    hs, ds, cs = sample
    m = monomial_vector(hs, ds)

    K = reduced_kernel(sample)
    rows = []

    for w in K.basis():
        for x in range(17):
            row = [F(0)] * DIM
            offset = x * ls
            for k in range(ls):
                row[offset + k] = m[k]
            rows.append(vector(F, row) * w[x])

    return rows

############################################
# 6. Intersect lifted constraints incrementally
############################################

def interpolate_kernel(samples):
    """
    Returns the kernel of the interpolation problem
    """
    constraints = []

    for i, sample in enumerate(samples):
        constraints.extend(lifted_constraints(sample))

        M = Matrix(F, constraints)
        K = M.right_kernel()

        print(f"After sample {i+1}: kernel dimension = {K.dimension()}")

        if K.dimension() == 0:
            return K

    return K

############################################
# 7. Example usage
############################################

# Collect samples
num_samples = 200
samples = [get_meta_sample() for _ in range(num_samples)]

K = interpolate_kernel(samples)

print("Final kernel dimension:", K.dimension())

In [ ]:
def get_inv(alphas):
    alpha0 = alphas[0]
    field = alpha0.parent()
    R.<q> = field[]
    [a0,a1,a2,a3,a4] = [alpha0] + [alpha/2 for alpha in alphas[1:]]
    f = field(1/4) * ((a0^4*a2^6 + 2*a0^4*a2^3*a4^3 + 32*a0*a1^3*a2^3*a4^3 + a0^4*a4^6)*q^6 + (-6*a0^4*a2^5*a3 + 24*a0^2*a1^2*a2^4*a4^2 - 6*a0^4*a2^2*a3*a4^3 - 96*a0*a1^3*a2^2*a3*a4^3 - 24*a0^2*a1^2*a2*a4^5)*q^5 + (15*a0^4*a2^4*a3^2 - 96*a0^2*a1^2*a2^3*a3*a4^2 + 6*a0^4*a2*a3^2*a4^3 + 96*a0*a1^3*a2*a3^2*a4^3 - 24*a0^3*a1*a2^2*a4^4 - 48*a1^4*a2^2*a4^4 + 24*a0^2*a1^2*a3*a4^5)*q^4 + (-20*a0^4*a2^3*a3^3 + 144*a0^2*a1^2*a2^2*a3^2*a4^2 - 2*a0^4*a2^3*a4^3 - 32*a0*a1^3*a2^3*a4^3 - 2*a0^4*a3^3*a4^3 - 32*a0*a1^3*a3^3*a4^3 + 48*a0^3*a1*a2*a3*a4^4 + 96*a1^4*a2*a3*a4^4 + 2*a0^4*a4^6 + 32*a0*a1^3*a4^6)*q^3 + (15*a0^4*a2^2*a3^4 - 96*a0^2*a1^2*a2*a3^3*a4^2 + 6*a0^4*a2^2*a3*a4^3 + 96*a0*a1^3*a2^2*a3*a4^3 - 24*a0^3*a1*a3^2*a4^4 - 48*a1^4*a3^2*a4^4 + 24*a0^2*a1^2*a2*a4^5)*q^2 + (-6*a0^4*a2*a3^5 + 24*a0^2*a1^2*a3^4*a4^2 - 6*a0^4*a2*a3^2*a4^3 - 96*a0*a1^3*a2*a3^2*a4^3 - 24*a0^2*a1^2*a3*a4^5)*q + a0^4*a3^6 + 2*a0^4*a3^3*a4^3 + 32*a0*a1^3*a3^3*a4^3 + a0^4*a4^6)
    C = HyperellipticCurve(f)
    return C.absolute_igusa_invariants_wamelen()

def get_inv2(alphas):
    alpha0 = alphas[0]
    field = alpha0.parent()
    R.<x> = field[]

    [a0,a1,a2,a3,a4] = [alpha/alpha0 for alpha in alphas]
    H3 = a4*(a2*x^2-a3*x-a1*a4)
    G3 = ( (a1^3*a4^3 + 3*a1*a2*a3*a4^4 + 2*a2^3*a4^3 + a2^3 + a3^3*a4^3)*x^3
          + (3*a1^2*a2*a4^5 - 3*a2^2*a3*a4^3 + 3*a1^2*a2*a4^2 - 3*a2^2*a3)*x^2
    + (-3*a1^2*a3*a4^5 + 3*a2*a3^2*a4^3 - 3*a1^2*a3*a4^2 + 3*a2*a3^2)*x
    + (-2*a1^3*a4^6 - a1^3*a4^3 + 3*a1*a2*a3*a4^4 + a2^3*a4^3 - a3^3)
         )
    lam3 = a1^3*a4^6 - 3*a1*a2*a3*a4^4 + a1^3*a4^3 - a2^3*a4^3 - a3^3*a4^3 - 3*a1*a2*a3*a4 - a2^3 - a3^3
    C = HyperellipticCurve(lam3*H3^3, G3)
    return C.absolute_igusa_invariants_wamelen()

map_list = Phi._maps

for k in range(1,9):
    map1 = map_list[3*k]
    map2 = map_list[3*(k+1)]
    
    cod = map1.codomain()
    dom = map2.domain()

    print(get_inv2(dom._h) == get_inv2(cod._h))

In [ ]:
def get_hyperelliptic(alphas):
    alpha0 = alphas[0]
    field = alpha0.parent()
    R.<x> = field[]

    [a0,a1,a2,a3,a4] = [alpha/alpha0 for alpha in alphas]
    H3 = a4*(a2*x^2-a3*x-a1*a4)
    G3 = ( (a1^3*a4^3 + 3*a1*a2*a3*a4^4 + 2*a2^3*a4^3 + a2^3 + a3^3*a4^3)*x^3
          + (3*a1^2*a2*a4^5 - 3*a2^2*a3*a4^3 + 3*a1^2*a2*a4^2 - 3*a2^2*a3)*x^2
    + (-3*a1^2*a3*a4^5 + 3*a2*a3^2*a4^3 - 3*a1^2*a3*a4^2 + 3*a2*a3^2)*x
    + (-2*a1^3*a4^6 - a1^3*a4^3 + 3*a1*a2*a3*a4^4 + a2^3*a4^3 - a3^3)
         )
    lam3 = a1^3*a4^6 - 3*a1*a2*a3*a4^4 + a1^3*a4^3 - a2^3*a4^3 - a3^3*a4^3 - 3*a1*a2*a3*a4 - a2^3 - a3^3
    C = HyperellipticCurve(lam3*H3^3, G3)
    return C

In [ ]:
get_hyperelliptic(dom._h).absolute_igusa_invariants_wamelen(), get_hyperelliptic(cod._h).absolute_igusa_invariants_wamelen()